# 17 — Sérsic-parameter recovery at scale

We reuse the same 5k HDF5 dataset and switch the target from the binary label to the **7-D Sérsic parameter vector**. The regressor is :class:`lensing.ml.models.SersicRegressor` (notebook 11) but trained at scale.

We measure the per-parameter recovery accuracy on a held-out split and produce a **reliability diagram**: a scatter of predicted vs. true with the 1:1 line and the 1σ scatter shaded — the standard plot in survey-scale ML papers.

In [ ]:
# Bootstrap: make `lensing` importable when running notebooks/ directly.
import sys
from pathlib import Path
repo = Path.cwd().resolve().parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

import lensing as gl
# Device-agnostic: prefer MPS (Apple GPU) → CUDA → CPU.
# Pass device="cpu" if you need to force the CPU path (e.g. for
# operators that have no MPS kernel yet, or for reproducibility).
device, dtype = gl.config.setup(seed=42)
print(f"using device: {device}")


In [ ]:
from pathlib import Path
import numpy as np
from torch.utils.data import DataLoader

DATA_PATH = Path('cache/lens_dataset_5k.h5')
assert DATA_PATH.exists(), 'Run notebook 16 first to generate the cache.'

# Same split as notebook 16 so comparisons are like-for-like.
n = gl.bigdata.HDF5Dataset(DATA_PATH).meta['n_samples'] if hasattr(
    gl.bigdata.HDF5Dataset(DATA_PATH).meta, '__getitem__') else 5000
rng = np.random.default_rng(123)
perm = rng.permutation(n)
n_train = int(0.7 * n); n_val = int(0.15 * n)
train_idx, val_idx, test_idx = perm[:n_train].tolist(), \
    perm[n_train:n_train + n_val].tolist(), \
    perm[n_train + n_val:].tolist()

train_ds = gl.bigdata.HDF5Dataset(DATA_PATH, target='params', indices=train_idx)
val_ds   = gl.bigdata.HDF5Dataset(DATA_PATH, target='params', indices=val_idx)
test_ds  = gl.bigdata.HDF5Dataset(DATA_PATH, target='params', indices=test_idx)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=128)


## 1. Train regressor

In [ ]:
model = gl.ml.models.SersicRegressor(in_channels=1, n_outputs=7)
history = gl.ml.train.fit_model(
    model, train_loader, val_loader,
    loss_fn=nn.MSELoss(),
    lr=1e-3, epochs=8,
    metrics={'mse': gl.ml.train.mse},
    log_every=1,
)
print(f'Trained in {history.duration_s:.1f}s')


## 2. Reliability diagrams

In [ ]:
from lensing.ml.datasets import PARAM_KEYS
test_loader = DataLoader(test_ds, batch_size=128)
preds, truths = [], []
model.eval()
with torch.no_grad():
    for x, y in test_loader:
        preds.append(model(x.to(device)).cpu().numpy())
        truths.append(y.numpy())
preds = np.vstack(preds); truths = np.vstack(truths)

fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for i, (ax, name) in enumerate(zip(axes.flatten(), PARAM_KEYS)):
    t = truths[:, i]; p = preds[:, i]
    ax.scatter(t, p, s=4, alpha=0.4, color='C0')
    lo, hi = t.min(), t.max()
    ax.plot([lo, hi], [lo, hi], 'k--', lw=0.8)
    # Robust scatter (1.4826 * MAD ≈ 1σ for a normal distribution)
    resid = p - t
    mad = np.median(np.abs(resid - np.median(resid)))
    sigma = 1.4826 * mad
    ax.set_title(f'{name}: σ_residual={sigma:.3f}')
    ax.set(xlabel=f'true {name}', ylabel=f'predicted {name}')
    ax.grid(alpha=0.3)
axes[1, 3].axis('off')
plt.tight_layout(); plt.show()


**Interpreting the σ_residual**: this is the *robust* (median-absolute-deviation) 1σ scatter of (pred − true). Rule of thumb: any σ_residual much smaller than the *prior* spread on that parameter is a useful constraint. Looking at the dataset priors (`Re ∈ [0.4, 1.5]`, `n ∈ [1.0, 6.0]`, etc.) we expect σ_Re ~ 0.05 arcsec and σ_n ~ 0.3 to be achievable; if the network does worse, more training data or a deeper backbone would help.

## 3. Comparison with classical Adam fit

On the same 100-sample test subset, we run the classical per-image Adam fit and compare the cumulative wall time and the per-parameter accuracy. The DNN's edge is **inference speed**: ~10–100× faster per image, at the cost of a small (but measurable) increase in scatter.

In [ ]:
import time

n_compare = 50
xy = gl.data.coordinate_grid(npix=test_ds.meta['npix'],
                              deltapix=test_ds.meta['deltapix'])
sigma_n = float(test_ds.meta['sigma'])

# DNN inference (one batch). Move both the input and the
# output across the device boundary because the trained
# `model` lives on `device` and downstream NumPy code
# expects CPU arrays.
t0 = time.perf_counter()
with torch.no_grad():
    imgs = torch.stack([test_ds[i][0] for i in range(n_compare)]).to(device)
    pred_dnn = model(imgs).cpu().numpy()
t_dnn = time.perf_counter() - t0

# Adam fit (one image at a time) — short budget
t0 = time.perf_counter()
pred_adam = []
for i in range(n_compare):
    img = test_ds[i][0][0]
    g = gl.light.Sersic(Ie=2., Re=1., n=2.5, x0=0., y0=0., e1=0., e2=0.)
    gl.inference.fit(
        g, xy, img,
        gl.inference.ReducedChiSquared(sigma=sigma_n, n_params=7),
        lr=0.05, epochs=200, grad_clip=10.0,
    )
    pred_adam.append([float(getattr(g, k)) for k in PARAM_KEYS])
pred_adam = np.array(pred_adam)
t_adam = time.perf_counter() - t0

true = np.stack([test_ds[i][1].numpy() for i in range(n_compare)])
err_dnn  = (pred_dnn  - true).std(axis=0)
err_adam = (pred_adam - true).std(axis=0)

print(f'DNN  : {t_dnn*1000:7.1f} ms total ({1000*t_dnn/n_compare:.2f} ms/sample)')
print(f'Adam : {t_adam*1000:7.1f} ms total ({1000*t_adam/n_compare:.2f} ms/sample)')
print(f'Speed-up: {t_adam/t_dnn:.0f}×')
print()
print('per-parameter std-of-residual (lower is better):')
for i, k in enumerate(PARAM_KEYS):
    print(f'  {k:<3s}: DNN = {err_dnn[i]:.3f}    Adam = {err_adam[i]:.3f}')
